In [0]:
%run ./00_Organizacao_do_Ambiente

bronze_schema: workspace.bronze
silver_schema: workspace.silver
gold_schema: workspace.gold
landing_path: /Volumes/workspace/rocket/cinedata_raw


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import Row

In [0]:
# Ver todos os valores distintos de status (pra saber exatamente que ruído tem: espaços, hífens, maiúsculas, lixo)
display(spark.sql(f"SELECT DISTINCT status, COUNT(*) as qtd FROM {bronze_schema}.tb_movies_info GROUP BY status ORDER BY status"))

status,qtd
null,66
"Floyd 'Money' Mayweather puts on a show no matter the opponent.""",1
his errant dad returns,1
sometimes referred to the Islamic Religious Police. All this,1
IN PRODUCTION,105
In Production,460
In-Production,46
PLANNED,3
POST PRODUCTION,93
Planned,42


In [0]:
# Ver uma amostra de datas, pra identificar os formatos diferentes presentes
display(spark.sql(f"SELECT DISTINCT release_date FROM {bronze_schema}.tb_movies_info LIMIT 40"))

release_date
2016-02-09
04-25-2018
2019-04-24
2019-10-01
2016-04-27
2018-02-13
2016-10-25
2017-07-05
2017-04-19
2016-08-03


In [0]:
from datetime import datetime

dq_results = []

def dq_check(nome_tabela: str, nome_check: str, df, condicao):
    total = df.count()
    falhas = df.filter(~condicao).count()
    passou = falhas == 0
    dq_results.append(Row(table_name=nome_tabela, check_name=nome_check, total_rows=total,
                           failed_rows=falhas, passed=passou, checked_at=datetime.now()))
    print(f"[{'PASS' if passou else 'FAIL'}] {nome_tabela} | {nome_check} | {falhas}/{total} falharam")

def dq_check_unique(nome_tabela: str, nome_check: str, df, colunas_chave: list):
    total = df.count()
    duplicadas = df.groupBy(*colunas_chave).count().filter("count > 1").count()
    passou = duplicadas == 0
    dq_results.append(Row(table_name=nome_tabela, check_name=nome_check, total_rows=total,
                           failed_rows=duplicadas, passed=passou, checked_at=datetime.now()))
    print(f"[{'PASS' if passou else 'FAIL'}] {nome_tabela} | {nome_check} | {duplicadas} chaves duplicadas de {total}")

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# dedup: mantém só o registro mais recente por id_filme, com base na ingestão
janela_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_silver_info = (
    df_bronze_info
    .withColumn("rn", F.row_number().over(janela_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
    # normaliza (remove hífen, maiúsculas) ANTES de traduzir; fora do domínio conhecido -> Não Informado
    .withColumn("status_normalizado", F.upper(F.trim(F.regexp_replace(F.col("status"), "-", " "))))
    .withColumn("status",
        F.when(F.col("status_normalizado") == "RELEASED", "Lançado")
         .when(F.col("status_normalizado") == "POST PRODUCTION", "Pós-Produção")
         .when(F.col("status_normalizado") == "IN PRODUCTION", "Em Produção")
         .when(F.col("status_normalizado") == "PLANNED", "Planejado")
         .when(F.col("status_normalizado") == "RUMORED", "Rumores")
         .when(F.col("status_normalizado") == "CANCELED", "Cancelado")
         .otherwise("Não Informado")
    )
    .drop("status_normalizado")
    # testa os 3 formatos observados na origem; o que não casar em nenhum vira NULL
       .withColumn("release_date",
        F.coalesce(
            F.try_to_date("release_date", "yyyy-MM-dd"),
            F.try_to_date("release_date", "MM-dd-yyyy"),
            F.try_to_date("release_date", "dd/MM/yyyy"),
        )
    )
    .withColumn("ano_lancamento", F.year("release_date"))
    .select("id", "title", "original_title", "release_date", "ano_lancamento",
            "runtime", "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("int"))
    .withColumn("idioma_original", F.when(F.col("idioma_original").rlike("^[a-z]{2}$"), F.col("idioma_original")))
)

dq_check_unique("silver.tb_info_filmes", "id_filme único", df_silver_info, ["id_filme"])
dq_check("silver.tb_info_filmes", "status_filme em domínio válido", df_silver_info,
         F.col("status_filme").isin(["Lançado", "Pós-Produção", "Em Produção", "Planejado", "Rumores", "Cancelado", "Não Informado"]))

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_silver_info.limit(10))

[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas de 97879
[PASS] silver.tb_info_filmes | status_filme em domínio válido | 0/97879 falharam


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
14564,Rings,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null
32471,Mixtape,Mixtape,2021-12-03,2021,94,en,Lançado,null,null
38258,Grizzly II: Revenge,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers
38492,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null
38700,Bad Boys for Life,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.
42018,The Horse Thief,盗马贼,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null
42330,Monkey Magic,大闹西游,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null
43074,Ghostbusters,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?
45033,20 Seconds of Joy,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null
46983,The Song of Styrene,Le Chant du styrène,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null


In [0]:
# 1. Deduplicação: linhas totais vs ids distintos na Bronze, comparado com a Silver
total_bronze = df_bronze_info.count()
ids_distintos_bronze = df_bronze_info.select("id").distinct().count()
total_silver = df_silver_info.count()

print(f"Bronze: {total_bronze} linhas | {ids_distintos_bronze} id_filme distintos")
print(f"Silver: {total_silver} linhas (esperado == {ids_distintos_bronze})")

# 2. Rename de colunas: confirma os nomes em português
df_silver_info.printSchema()

# 3. Qualidade do conteúdo: só devem sobrar os 7 status válidos + "Não Informado"
display(df_silver_info.select("status_filme").distinct())

# 4. Coerência da coluna derivada: ano_lancamento deve bater com o ano de data_lancamento
inconsistentes = df_silver_info.filter(
    F.col("data_lancamento").isNotNull() & (F.year("data_lancamento") != F.col("ano_lancamento"))
).count()
print(f"Inconsistências ano vs data: {inconsistentes} (esperado 0)")

Bronze: 106930 linhas | 97879 id_filme distintos
Silver: 97879 linhas (esperado == 97879)
root
 |-- id_filme: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)



status_filme
Lançado
Pós-Produção
Não Informado
Em Produção
Planejado


Inconsistências ano vs data: 0 (esperado 0)


In [0]:
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# agrega por dia (a API pode trazer mais de uma cotação no mesmo dia)
df_cotacao_diaria = (
    df_bronze_cotacao
    .withColumn("data_cotacao", F.to_date(F.substring("dataHoraCotacao", 1, 10), "yyyy-MM-dd"))
    .groupBy("data_cotacao")
    .agg(F.max("cotacaoCompra").alias("cotacao_dolar"))
)

data_min = df_cotacao_diaria.agg(F.min("data_cotacao")).first()[0]

# calendário contínuo: do primeiro dia com cotação real até hoje
df_calendario = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(data_min), F.current_date(), F.expr("interval 1 day"))).alias("data_cotacao"))
)

janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacao_diaria, on="data_cotacao", how="left")
    # forward-fill: pega o último valor não nulo visto até a data atual
    .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(janela_ffill))
    .orderBy("data_cotacao")
)

dq_check("silver.tb_cotacao_dolar", "cotacao_dolar sem nulos após forward-fill", df_silver_cotacao,
         F.col("cotacao_dolar").isNotNull())

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_silver_cotacao)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | cotacao_dolar sem nulos após forward-fill | 0/8 falharam


data_cotacao,cotacao_dolar
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569
2026-09-19,5.1569
2026-09-20,5.1569
2026-09-21,5.1569


In [0]:
spark.table(f"{bronze_schema}.tb_movies_financials").printSchema()

root
 |-- id: integer (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Amostra de valores "estranhos" (não puramente numéricos) em budget/revenue
display(spark.sql(f"""
    SELECT budget, revenue
    FROM {bronze_schema}.tb_movies_financials
    WHERE budget NOT RLIKE '^[0-9]+(\\\\.[0-9]+)?$'
       OR revenue NOT RLIKE '^[0-9]+(\\\\.[0-9]+)?$'
       OR budget IS NULL OR revenue IS NULL
    LIMIT 40
"""))

budget,revenue
58000000,Unknown
250000000,Não Informado
$ 97000000,619021436
$ 175000000,-800526015
$ 250000000,Não Informado
110000000,Unknown
USD 150000000,527000000
116000000,-856085151
34.0M,226945087
175000000,Não Informado


In [0]:
def limpar_valor_monetario(nome_coluna):
    valor_bruto = F.trim(F.col(nome_coluna))
    tokens_ausentes = ["UNKNOWN", "NÃO INFORMADO", "N/A", ""]
    eh_ausente = F.upper(valor_bruto).isin(tokens_ausentes)

    valor_limpo = F.regexp_replace(valor_bruto, r"(?i)USD|\$|,|\s", "")

    # sufixos de escala: K = mil, M = milhão (ex.: "34.0K" -> 34000, "34.0M" -> 34000000)
    eh_milhar = F.upper(valor_limpo).rlike(r"^[0-9]+\.?[0-9]*K$")
    eh_milhoes = F.upper(valor_limpo).rlike(r"^[0-9]+\.?[0-9]*M$")
    numero_sem_sufixo = F.regexp_replace(valor_limpo, r"(?i)[KM]$", "")

    valor_numerico = (
        F.when(eh_milhar, numero_sem_sufixo.try_cast("double") * 1_000)
         .when(eh_milhoes, numero_sem_sufixo.try_cast("double") * 1_000_000)
         .otherwise(valor_limpo.try_cast("double"))
    )

    return F.when(eh_ausente, F.lit(None).cast("double")).otherwise(valor_numerico)

In [0]:
df_bronze_fin = spark.table(f"{bronze_schema}.tb_movies_financials")

# mesma fonte fragmentada de tb_movies_info: dedup pra garantir 1 linha por filme
janela_dedup_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

# cotação única (mais recente) aplicada a todos os filmes — ver decisão registrada na 2.7
cotacao_atual = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc())
    .select("cotacao_dolar").first()["cotacao_dolar"]
)

df_silver_fin = (
    df_bronze_fin
    .withColumn("rn", F.row_number().over(janela_dedup_fin))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("orcamento_usd", limpar_valor_monetario("budget"))
    .withColumn("receita_usd", limpar_valor_monetario("revenue"))
    # valor zerado ou negativo não é um orçamento/receita real -> ausente
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") <= 0, None).otherwise(F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") <= 0, None).otherwise(F.col("receita_usd")))
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(cotacao_atual), 2))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(cotacao_atual), 2))
    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn("margem_lucro_percentual",
        F.when(F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") != 0),
               F.round(F.col("lucro_usd") / F.col("orcamento_usd") * 100, 2))
    )
    .select("id", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_percentual")
    .withColumnRenamed("id", "id_filme")
)

dq_check_unique("silver.tb_financeiro_filmes", "id_filme único", df_silver_fin, ["id_filme"])
dq_check("silver.tb_financeiro_filmes", "orcamento_usd > 0 quando não nulo", df_silver_fin,
         F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0))

df_silver_fin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

display(df_silver_fin.limit(15))

[PASS] silver.tb_financeiro_filmes | id_filme único | 0 chaves duplicadas de 99006
[PASS] silver.tb_financeiro_filmes | orcamento_usd > 0 quando não nulo | 0/99006 falharam


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
14564,2.5E7,8.308089E7,1.289225E8,4.2843984164E8,5.808089E7,2.9951734164E8,232.32
32471,null,null,null,null,null,null,null
38258,7500000.0,null,3.867675E7,null,null,null,null
38492,null,null,null,null,null,null,null
38700,9.0E7,4.26505244E8,4.64121E8,2.19944489278E9,3.36505244E8,1.7353238927800002E9,373.89
42018,null,null,null,null,null,null,null
42330,null,null,null,null,null,null,null
43074,1.44E8,2.29147509E8,7.425936E8,1.18169078916E9,8.5147509E7,4.390971891600001E8,59.13
45033,337200.0,null,1738906.68,null,null,null,null
46983,null,null,null,null,null,null,null


In [0]:
# Análise dos formatos dos dados na camada bronze
spark.table(f"{bronze_schema}.tb_movies_metrics").printSchema()

display(spark.table(f"{bronze_schema}.tb_movies_metrics").limit(30))

# valores de popularidade que não são puramente numéricos (formatação inconsistente)
display(spark.sql(f"""
    SELECT DISTINCT popularity
    FROM {bronze_schema}.tb_movies_metrics
    WHERE popularity NOT RLIKE '^[0-9]+\\\\.?[0-9]*$'
    LIMIT 30
"""))

root
 |-- id: integer (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-21T15:11:38.774Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-21T15:11:38.774Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-21T15:11:38.774Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-21T15:11:38.774Z
271110,70.741,7.4,21541,7.8,947222,2026-09-21T15:11:38.774Z
284054,43.665,7.39,null,7.3,924922,2026-09-21T15:11:38.774Z
284052,70.535,7.427,20935,7.5,895880,2026-09-21T15:11:38.774Z
315635,65.88,7.345,20507,7.4,835116,2026-09-21T15:11:38.774Z
283995,67.553,7.624,20353,7.6,844767,2026-09-21T15:11:38.774Z
297761,35.356,5.909,20097,5.9,null,2026-09-21T15:11:38.774Z


popularity
"154,34"
"54,522"
"54,628"
"44,51"
"98,11"
"50,399"
"50,088"
"causing others from across the Spider-Verse to be inadvertently transported to his dimension."""
"38,002"
"52,471"


In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

janela_dedup_metrics = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

def limpar_nota(nome_coluna):
    # vírgula como separador decimal inconsistente + cast seguro (column shift joga texto/número fora de escala aqui)
    valor = F.regexp_replace(F.trim(F.col(nome_coluna)), ",", ".").try_cast("double")
    # fora de 0-10 (inclui erro de escala, ex: 79.97) -> ausente
    return F.when(valor.between(0, 10), valor)

def limpar_contagem(nome_coluna):
    valor = F.trim(F.col(nome_coluna)).try_cast("int")
    return F.when(valor >= 0, valor)  # negativo é inválido -> ausente

df_silver_metricas = (
    df_bronze_metrics
    .withColumn("rn", F.row_number().over(janela_dedup_metrics))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("popularity", F.regexp_replace(F.trim(F.col("popularity")), ",", ".").try_cast("double"))
    .withColumn("popularity", F.when(F.col("popularity") >= 0, F.col("popularity")))
    .withColumn("vote_average", limpar_nota("vote_average"))
    .withColumn("averageRating", limpar_nota("averageRating"))
    .withColumn("vote_count", limpar_contagem("vote_count"))
    .withColumn("numVotes", limpar_contagem("numVotes"))
    .select("id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

dq_check_unique("silver.tb_metricas_engajamento", "id_filme único", df_silver_metricas, ["id_filme"])
dq_check("silver.tb_metricas_engajamento", "notas entre 0 e 10 quando não nulas", df_silver_metricas,
         (F.col("nota_media_tmdb").isNull() | F.col("nota_media_tmdb").between(0, 10)) &
         (F.col("nota_media_imdb").isNull() | F.col("nota_media_imdb").between(0, 10)))

df_silver_metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_metricas.limit(15))

[PASS] silver.tb_metricas_engajamento | id_filme único | 0 chaves duplicadas de 99013
[PASS] silver.tb_metricas_engajamento | notas entre 0 e 10 quando não nulas | 0/99013 falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
14564,24.584,4.966,2375,null,46286
32471,8.929,7.064,118,6.6,4617
38258,null,3.161,28,null,null
38492,2.98,7.1,null,7.8,212
38700,46.619,7.139,7570,6.5,199420
42018,3.403,6.63,27,6.8,1750
42330,1.561,10.0,1,6.4,34
43074,40.052,5.371,5976,null,260417
45033,0.6,8.0,2,null,154
46983,0.693,8.5,2,7.0,1209


In [0]:
# Análise da tb_movies_reviews antes da limpeza: preciso ver o schema,
# uma amostra real e os casos fora da regra de negócio (nota fora de 0-10, comentário vazio)
# pra desenhar o tratamento em cima do que existe de fato na base, não do que eu imagino.
spark.table(f"{bronze_schema}.tb_movies_reviews").printSchema()

display(spark.table(f"{bronze_schema}.tb_movies_reviews").limit(30))

display(spark.sql(f"""
    SELECT nota, comentario
    FROM {bronze_schema}.tb_movies_reviews
    WHERE nota NOT RLIKE '^[0-9]+\\\\.?[0-9]*$'
       OR trim(comentario) = ''
       OR comentario IS NULL
    LIMIT 30
"""))

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: double (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-21T15:11:48.923Z
637007,Lucas Reis 602,3.9,null,2026-09-21T15:11:48.923Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-21T15:11:48.923Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-21T15:11:48.923Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-21T15:11:48.923Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-09-21T15:11:48.923Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-09-21T15:11:48.923Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-09-21T15:11:48.923Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-09-21T15:11:48.923Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-09-21T15:11:48.923Z


nota,comentario
4.4,null
3.9,null
7.5,null
0.0,null
5.6,null
null,null
3.5,null
3.5,null
0.2,null
null,null


In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_avaliacoes = (
    df_bronze_reviews
    # duplicata exata (mesmo filme+usuário+nota+comentário) -> mantém só 1 ocorrência
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    # nota fora de 0-10 -> ausente
    .withColumn("nota", F.when(F.col("nota").between(0, 10), F.col("nota")))
    # comentário vazio, só espaço ou nulo -> texto padronizado
    .withColumn("comentario", F.coalesce(F.nullif(F.trim(F.col("comentario")), F.lit("")), F.lit("Sem comentário")))
    .select("id", "nome", "nota", "comentario")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

dq_check("silver.tb_avaliacoes_usuarios", "nota_usuario entre 0 e 10 quando não nula", df_silver_avaliacoes,
         F.col("nota_usuario").isNull() | F.col("nota_usuario").between(0, 10))
dq_check("silver.tb_avaliacoes_usuarios", "comentario_usuario nunca vazio/nulo", df_silver_avaliacoes,
         F.col("comentario_usuario").isNotNull() & (F.trim(F.col("comentario_usuario")) != ""))

df_silver_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_avaliacoes.limit(15))

[PASS] silver.tb_avaliacoes_usuarios | nota_usuario entre 0 e 10 quando não nula | 0/32412 falharam
[PASS] silver.tb_avaliacoes_usuarios | comentario_usuario nunca vazio/nulo | 0/32412 falharam


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


In [0]:
# Análise de tb_credits_and_tags (coluna genres) antes de tratar separadores/column shift
spark.table(f"{bronze_schema}.tb_credits_and_tags").printSchema()
display(spark.table(f"{bronze_schema}.tb_credits_and_tags").select("id", "genres").limit(30))

root
 |-- id: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,genres
293660,"Action, Adventure, Comedy"
299536,"Adventure, Action, Science Fiction"
299534,"Adventure, Science Fiction, Action"
475557,"Crime, Thriller, Drama"
271110,"Adventure, Action, Science Fiction"
284054,"Action, Adventure, Science Fiction"
284052,"Action, Adventure, Fantasy"
315635,"Action, Adventure, Science Fiction, Drama"
283995,"Science Fiction, Adventure, Action"
297761,Action|Adventure|Fantasy


In [0]:
GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_generos_explodido = (
    df_bronze_credits
    .select("id", "genres")
    # normaliza os 3 separadores encontrados (vírgula, pipe, ponto-e-vírgula) antes do split
    .withColumn("genres_normalizado", F.regexp_replace(F.col("genres"), r"[|;]", ","))
    .withColumn("genero", F.explode(F.split(F.col("genres_normalizado"), ",")))
    # remove aspas/barras invertidas residuais (sobra de escaping do CSV) antes de comparar
    .withColumn("genero", F.trim(F.regexp_replace(F.col("genero"), r'["\\]', "")))
    # domínio fechado do TMDB: qualquer coisa fora da lista oficial é lixo de column shift
    # (caminho de imagem, texto de sinopse, número de popularidade vazado)
    .filter(F.col("genero").isin(GENEROS_VALIDOS))
    .select("id", "genero")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("genero", "nome_genero")
    .dropDuplicates(["id_filme", "nome_genero"])
)

display(df_generos_explodido.select("nome_genero").distinct().orderBy("nome_genero"))

nome_genero
Action
Adventure
Animation
Comedy
Crime
Documentary
Drama
Family
Fantasy
History


In [0]:
dq_check("silver.tb_generos", "nome_genero dentro da lista oficial TMDB", df_generos_explodido,
         F.col("nome_genero").isin(GENEROS_VALIDOS))

df_generos_explodido.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

print(f"silver.tb_generos: {df_generos_explodido.count()} linhas (filme x gênero)")

[PASS] silver.tb_generos | nome_genero dentro da lista oficial TMDB | 0/142160 falharam
silver.tb_generos: 142160 linhas (filme x gênero)


In [0]:
TOKENS_AUSENTES_E_IDIOMAS = [
    "N/A", "NONE", "UNKNOWN", "ENGLISH", "SPANISH", "FRENCH", "GERMAN", "ITALIAN",
    "PORTUGUESE", "JAPANESE", "KOREAN", "CHINESE", "MANDARIN", "CANTONESE", "RUSSIAN",
    "HINDI", "ARABIC", "SWEDISH", "NORWEGIAN", "DANISH", "DUTCH", "POLISH", "TURKISH",
    "GREEK", "HEBREW", "THAI", "VIETNAMESE", "INDONESIAN", "FILIPINO", "TAGALOG",
    "CZECH", "HUNGARIAN", "FINNISH", "ROMANIAN", "UKRAINIAN"
]

def extrair_entidades(df, coluna, tipo_entidade):
    return (
        df
        .select("id", coluna)
        .withColumn("valor_normalizado", F.regexp_replace(F.col(coluna), r"[|;]", ","))
        .withColumn("item", F.explode(F.split(F.col("valor_normalizado"), ",")))
        .withColumn("item", F.trim(F.regexp_replace(F.col("item"), r'["\\\[\]]', "")))
        .filter(
            (F.col("item") != "") &
            (~F.upper(F.col("item")).isin(TOKENS_AUSENTES_E_IDIOMAS)) &
            (F.length("item") <= 60) &
            (~F.col("item").rlike("^[0-9.]+$")) &
            (~F.col("item").rlike("(?i)^/.*\\.(jpg|png)$")) &
            (~F.col("item").rlike("\\.$")) &                # frase vazada termina em ponto final
            (F.col("item").rlike("^[A-ZÀ-Ý0-9]"))            # nome de entidade começa com maiúscula/número
        )
        .withColumn("nome_entidade",
            F.when(F.col("item") == F.upper(F.col("item")), F.initcap(F.col("item")))
             .when(F.col("item") == F.lower(F.col("item")), F.initcap(F.col("item")))
             .otherwise(F.col("item"))
        )
        .select("id", "nome_entidade", F.lit(tipo_entidade).alias("tipo_entidade"))
    )

df_pessoas_empresas = (
    extrair_entidades(df_bronze_credits, "cast", "Ator")
    .unionByName(extrair_entidades(df_bronze_credits, "directors", "Diretor"))
    .unionByName(extrair_entidades(df_bronze_credits, "writers", "Roteirista"))
    .unionByName(extrair_entidades(df_bronze_credits, "production_companies", "Produtora"))
    .withColumnRenamed("id", "id_filme")
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

dq_check("silver.tb_pessoas_empresas", "tipo_entidade em domínio válido", df_pessoas_empresas,
         F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista", "Produtora"]))

df_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

print(f"silver.tb_pessoas_empresas: {df_pessoas_empresas.count()} linhas")

[PASS] silver.tb_pessoas_empresas | tipo_entidade em domínio válido | 0/885074 falharam
silver.tb_pessoas_empresas: 885074 linhas
